# K-Means - Clustering

Os dados normalizados serão preparados selecionando variáveis relevantes (ex.: pH, condutividade elétrica, nitrato). 

O algoritmo será aplicado para agrupar os dados por similaridade, testando n_clusters de 2 a 10, definido pelo método do cotovelo (Kaufman; Russeeuw, 1990). 

O objetivo é identificar padrões espaciais ou temporais (ex.: regiões com alta turbidez) para manejo hídrico. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pcj.utils import BASE_DIR, dados

In [ ]:
# Dados para testes

dados_limpos, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")

X = dados_limpos.copy()

In [ ]:
# Verificações

# print(dados_limpos['Data'].dtype, end='')
# print(dados_limpos['data_normalizada'].dtype)
# print(dados_limpos.columns)
# print(variaveis)
print(dados_limpos[variaveis])

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 01 - Minha Tentativa

In [ ]:
# Primeira tentativa

from sklearn.cluster import KMeans

# Salvar cópia das colunas das variáveis numa variável
X = dados_limpos[variaveis].copy()

# Eliminar células vaziass
Xnonan = X.dropna()

# kmeans = KMeans(n_clusters=2, random_state=0, n_init="auto").fit(Xnonan)
print(X)

In [ ]:
# Setup

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Salvar cópia das colunas das variáveis numa variável
X = dados_limpos[variaveis].copy()

# Eliminar células vaziass
X_nonan = X.dropna(axis=0, how='any')
# Salvar o Index do DataFrame sem células vazias numa variável
idx_nonan = X_nonan.index

scaler = StandardScaler()   # Escalonamento = Normalização dos dados
X_scaled = scaler.fit_transform(X_nonan)

pca = PCA(n_components=0.95, random_state=0)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# Teste

kmeans = KMeans(n_clusters=2, random_state=0)
cluster_labels = kmeans.fit_predict(X_nonan)
distancias = np.linalg.norm(X_nonan - kmeans.cluster_centers_[cluster_labels], axis=1)

kmeans.labels_

# distancias

In [ ]:
# Teste

kmeans = KMeans(n_clusters=2, random_state=0)
cluster_labels = kmeans.fit_predict(X_pca)
distancias = np.linalg.norm(X_pca - kmeans.cluster_centers_[cluster_labels], axis=1)

kmeans.labels_

# distancias

In [ ]:
n_clusters_list = [2, 3, 4]

fig, axs = plt.subplots(
    1,
    len(n_clusters_list),
    figsize=(20, 10)
)

axs = axs.T

for i, n_clusters in enumerate(n_clusters_list):
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init="auto")
    kmeans.fit(X_pca)
    centers = kmeans.cluster_centers_

    axs[i].scatter(X_pca[:, 0], X_pca[:, 1], s=10, c=kmeans.labels_)
    axs[i].scatter(centers[:, 0], centers[:, 1], c="r", s=20)

    axs[i].set_title(f"{kmeans} : {n_clusters} clusters")

for ax in axs.flat:
    ax.label_outer()
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()

plt.show()

In [ ]:
X_nonan

In [ ]:
# Teste

n_clusters = 3  # De acordo com os testes anteriores, 3 parece ser o melhor número de clusters

kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init="auto")
kmeans.fit(X_nonan)  # Talvez trocar X_nonan para X_pca
labels = kmeans.labels_

# Extrai datas que correspondem com as linhas de idx_nonan (algumas são removidas devido às células vazias)
dates = dados_limpos.loc[idx_nonan, 'data_normalizada']

# Subplots individuais para as variáveis
fig, axs = plt.subplots(len(variaveis), 1, figsize=(15, 4*len(variaveis)))

if len(variaveis) == 1:
    axs = [axs]

for i, var in enumerate(variaveis):
    scatter = axs[i].scatter(dates, X_nonan[var], c=labels, cmap='viridis', s=20)
    axs[i].set_ylabel(var)
    axs[i].set_xlabel('Data')
    axs[i].set_title(f'{var} ao longo do tempo (colorido segundo clustering K-Means)')
    axs[i].tick_params(axis='x', rotation=45)

cbar_ax = fig.add_axes([0.92, 0.1, 0.02, 0.8])
cbar = fig.colorbar(scatter, cax=cbar_ax, label='Cluster')
cbar.set_ticks(range(n_clusters))

plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.show()